# Graph population statistics

QC for the production blob graphs under `{OUT_ROOT}/{split}/{galaxy_id}/graphs/`.
Re-run as Slurm arrays finish — missing galaxies just show up as not-done in the coverage grid.

Each graph is one `(cell, GMM, sky-part)` after the frozen CMD slab, $\pi<1$, $k=64$, $w\le 1.65$, and $N\le 8000$ sky split. Empty blobs (zero mocks) are the negative class.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

from cpen.apps.streams.preprocess.config import DEFAULT_PIPELINE_ROOT
from cpen.apps.streams.preprocess.gmm import BLOB_N_MAX
from cpen.apps.streams.preprocess.graph_qc import (
    load_graph_manifests,
    pipeline_coverage,
    plot_graph_coverage,
    plot_graph_population,
    plot_graphs_per_galaxy,
    print_graph_population,
)

try:
    import scienceplots  # noqa: F401
    plt.style.use(["science", "no-latex"])
except Exception:
    pass
mpl.rcParams.update({"figure.dpi": 140, "savefig.dpi": 200})

OUT_ROOT = Path(DEFAULT_PIPELINE_ROOT)
print(f"OUT_ROOT={OUT_ROOT}  exists={OUT_ROOT.is_dir()}")


## Coverage

Green = stage wrote `.done`. Use this while jobs are still in the queue.


In [ ]:
coverage = pipeline_coverage(OUT_ROOT)
print_graph_population(pd.DataFrame(), coverage)
display(coverage.drop(columns=["path"], errors="ignore"))
plot_graph_coverage(coverage)


## Size, occupancy, edge labels

Loaded from `graphs/manifest.parquet` (plus blob `width` when stage 3 is present). No need to open the `.npz` files.


In [ ]:
tab = load_graph_manifests(OUT_ROOT)
print_graph_population(tab, coverage)
plot_graph_population(tab, n_max=BLOB_N_MAX)
plot_graphs_per_galaxy(tab)


## Per-galaxy table and tails


In [ ]:
if tab is None or not len(tab):
    print("No graph manifests yet — re-run this cell after stage 4.")
else:
    gal = (
        tab.groupby(["split", "galaxy_id"], sort=False)
        .agg(
            n_graphs=("n_stars", "size"),
            n_empty=("is_empty", "sum"),
            median_N=("n_stars", "median"),
            max_N=("n_stars", "max"),
            n_mock=("n_mock", "sum"),
            knn_pos=("n_knn_pos", "sum"),
            n_knn=("n_knn", "sum"),
            hyper_pos=("n_hyper_pos", "sum"),
            n_hyper=("n_hyper", "sum"),
        )
        .reset_index()
    )
    gal["empty_frac"] = gal.n_empty / gal.n_graphs
    gal["knn_pos_frac"] = gal.knn_pos / gal.n_knn.replace(0, pd.NA)
    gal["hyper_pos_frac"] = gal.hyper_pos / gal.n_hyper.replace(0, pd.NA)
    display(gal.sort_values(["split", "galaxy_id"]))

    print("\n===== largest graphs =====")
    cols = [c for c in (
        "split", "galaxy_id", "cell_id", "gmm_label", "blob_part",
        "n_stars", "n_mock", "mock_frac", "width", "n_knn", "n_hyper",
    ) if c in tab.columns]
    display(tab.sort_values("n_stars", ascending=False)[cols].head(12))
    print("\n===== mock-richest graphs =====")
    display(tab.sort_values(["n_mock", "n_stars"], ascending=False)[cols].head(12))

    n_over = int((tab.n_stars > BLOB_N_MAX).sum())
    print(f"\nN > {BLOB_N_MAX:,}: {n_over}  (expect 0 after the sky split)")
